In [ ]:
pip install part 

In [ ]:
import part 

In [ ]:
# ==========================================================
# Cell 3 : Read Evaluation Dataset 
# ==========================================================

from pyspark.sql import functions as F
from databricks.sdk.runtime import dispaly

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

RAG_EVAL_TABLE = "knwbt_airr_foundry_dev_catalog.secured.rag_eval_set"

TOP_K_XML = 50

# ----------------------------------------------------------
# Read Evaluation Dataset
# ----------------------------------------------------------

rag_eval_df = (

    spark.table(RAG_EVAL_TABLE)

)

print("=" * 80)
print("RAG EVALUATION DATASET")
print("=" * 80)

print(f"Total Questions : {rag_eval_df.count()}")

# ----------------------------------------------------------
# Select Required Columns
# ----------------------------------------------------------

rag_eval_df = (

    rag_eval_df

    .select(

        "doc_id",
        "request",
        "question_type",
        "difficulty",
        "ground_truth",
        "source_doc_ref",
        "source_doc_title",
        "source",
        "expected_facts",
        "retrieved_context"

    )

)

# ----------------------------------------------------------
# Take Top 50 XML Sources
# ----------------------------------------------------------

top50_eval = (

    rag_eval_df

    .limit(TOP_K_XML)

)

# ----------------------------------------------------------
# Remove Duplicate XML Paths
# ----------------------------------------------------------

top50_sources = (

    top50_eval

    .select("source")

    .dropDuplicates()

)

print()

print("=" * 80)
print("TOP XML SOURCES")
print("=" * 80)

print(f"Questions Selected : {top50_eval.count()}")

print(f"Unique XML Sources : {top50_sources.count()}")

print()

display(top50_eval)

print()

display(top50_sources)






In [ ]:
# ==========================================================
# Cell 4 : Extract Matching XML Documents
# ==========================================================

from pyspark.sql import functions as F

# ----------------------------------------------------------
# Read Raw XML Table
# ----------------------------------------------------------

RAW_TABLE = "knwbt_airr_foundry_dev_catalog.secured.knowbot_raw_data"

raw_df = spark.table(RAW_TABLE)

print("=" * 80)
print("KNOWBOT RAW DATA")
print("=" * 80)

print(f"Total XML Documents : {raw_df.count()}")

# ----------------------------------------------------------
# Match XML Paths
# ----------------------------------------------------------

matched_xml = (

    raw_df.alias("raw")

    .join(

        top50_sources.alias("eval"),

        F.col("raw.source") == F.col("eval.source"),

        "inner"

    )

)

# ----------------------------------------------------------
# Remove Duplicate XML Documents
# ----------------------------------------------------------

matched_xml = (

    matched_xml

    .dropDuplicates(["doc_id"])

)

# ----------------------------------------------------------
# Select Required Columns
# ----------------------------------------------------------

matched_xml = (

    matched_xml

    .select(

        "doc_id",

        "title",

        "content",

        "reference",

        "source",

        "version",

        "content_update_date",

        "content_author"

    )

)

print()

print("=" * 80)
print("MATCHED XML DOCUMENTS")
print("=" * 80)

print(f"Matched XML Documents : {matched_xml.count()}")

print()

display(matched_xml)

In [ ]:
# ==========================================================
# Cell 5 : XML Data Cleaning
# ==========================================================

from pyspark.sql import functions as F

print("=" * 80)
print("XML DATA CLEANING")
print("=" * 80)

# ----------------------------------------------------------
# Input from Cell 4
# ----------------------------------------------------------

xml_df = matched_xml

print(f"Matched XML Documents : {xml_df.count()}")

# ----------------------------------------------------------
# Remove Duplicate XML Documents
# ----------------------------------------------------------

xml_df = (

    xml_df

    .dropDuplicates(

        ["doc_id"]

    )

)

# ----------------------------------------------------------
# Replace NULL Content
# (Do NOT remove XML documents)
# ----------------------------------------------------------

xml_df = (

    xml_df

    .fillna(

        {

            "content": ""

        }

    )

)

# ----------------------------------------------------------
# Clean XML Content
# ----------------------------------------------------------

xml_df = (

    xml_df

    .withColumn(

        "content",

        F.regexp_replace(

            F.col("content"),

            r"\s+",

            " "

        )

    )

)

# ----------------------------------------------------------
# Trim Text Columns
# ----------------------------------------------------------

xml_df = (

    xml_df

    .withColumn(

        "title",

        F.trim(

            F.col("title")

        )

    )

    .withColumn(

        "reference",

        F.trim(

            F.col("reference")

        )

    )

    .withColumn(

        "source",

        F.trim(

            F.col("source")

        )

    )

)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print("=" * 80)
print("CLEAN XML DATA")
print("=" * 80)

print(f"XML Documents Ready : {xml_df.count()}")

print()


In [ ]:
# ==========================================================
# Stage 4 : Knowledge Node Creation
# Cell 6 : Create One Node per XML  (deterministic + normalized)
# ==========================================================
from pyspark.sql import functions as F
from pyspark.sql import Window

print("=" * 80)
print("KNOWLEDGE NODE CREATION")
print("=" * 80)

# ----------------------------------------------------------
# Input from Cell 5
# ----------------------------------------------------------
node_df = xml_df
print(f"XML Documents (raw) : {node_df.count()}")

# ----------------------------------------------------------
# Reference normalization
# MUST be identical to the normalization used in Cell 7
# (hyperlink extraction). If these diverge -> orphan edges.
# ----------------------------------------------------------
def normalize_ref_col(c):
    c = F.trim(c)
    # strip a leading "xwiki:" if present in some refs
    c = F.regexp_replace(c, r"^xwiki:", "")
    # collapse internal whitespace (rare but happens in exports)
    c = F.regexp_replace(c, r"\s+", "")
    return c

# ----------------------------------------------------------
# Build base node projection
# ----------------------------------------------------------
base = (
    node_df
    .select(
        F.col("reference").alias("node_id"),
        F.col("title").alias("node_name"),
        F.col("content").alias("node_content"),
        F.col("doc_id"),
        F.col("source"),
        F.col("version"),
        F.col("content_update_date"),
        F.col("content_author"),
    )
    .withColumn("node_id", normalize_ref_col(F.col("node_id")))
    # length used both for dedup ranking and content flag
    .withColumn(
        "content_len",
        F.length(F.coalesce(F.col("node_content"), F.lit("")))
    )
)

# ----------------------------------------------------------
# Deterministic dedup:
# per node_id keep the row with the most content,
# tie-break on newest update date.
# ----------------------------------------------------------
w = (
    Window
    .partitionBy("node_id")
    .orderBy(
        F.col("content_len").desc_nulls_last(),
        F.col("content_update_date").desc_nulls_last(),
    )
)

knowledge_nodes = (
    base
    .withColumn("_rk", F.row_number().over(w))
    .filter(F.col("_rk") == 1)
    .drop("_rk")
    # flag stub / empty-content nodes (valid as link targets,
    # but retriever should not treat them as answer-bearing)
    .withColumn("has_content", F.col("content_len") > 0)
    .cache()
)

# ----------------------------------------------------------
# Node Statistics
# ----------------------------------------------------------
total_nodes    = knowledge_nodes.count()   # materializes cache
n_with_content = knowledge_nodes.filter(F.col("has_content")).count()
n_stub         = total_nodes - n_with_content

print()
print("=" * 80)
print("KNOWLEDGE NODES")
print("=" * 80)
print(f"Total Nodes Created : {total_nodes}")
print(f"With Content        : {n_with_content}")
print(f"Stub / Empty        : {n_stub}")
print()
display(knowledge_nodes)

In [ ]:
# ==========================================================
# Stage 5 : Hyperlink Extraction
# Cell 7 : Extract XML Hyperlinks (generic · 20k-safe)
# ==========================================================
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

print("=" * 80)
print("HYPERLINK EXTRACTION")
print("=" * 80)

# ----------------------------------------------------------
# Normalize XML Reference  (same logic as yours)
# ----------------------------------------------------------
def normalize_reference(ref):
    if not ref:
        return None
    ref = ref.strip()
    if "||" in ref:
        ref = ref.split("||")[0]
    if "#" in ref:
        ref = ref.split("#")[0]
    ref = ref.replace("&amp;", "&")
    ref = re.sub(r"\s+", "", ref)      # refs have no spaces
    return ref.strip()

# ----------------------------------------------------------
# Parse one [[...]] match -> (link_type, target_reference)
# Handles doc:wiki: , doc: , AND bare Temp_Import (your data has all 3)
# ----------------------------------------------------------
def parse_target(target):
    if not target:
        return None
    target = target.strip()

    if target.startswith("doc:wiki:"):
        return ("doc:wiki", normalize_reference(target[len("doc:wiki:"):]))

    if target.startswith("doc:Temp_Import"):
        return ("doc:Temp_Import", normalize_reference(target[len("doc:"):]))

    # BARE form: [[Label>>Temp_Import.<...>.WebHome]] (no doc: prefix)
    if target.startswith("Temp_Import"):
        return ("bare:Temp_Import", normalize_reference(target))

    return None

# ----------------------------------------------------------
# UDF: extract all links from one document's content.
# Runs in EXECUTORS (distributed) — no toLocalIterator, no driver list.
# Returns array of structs.
# ----------------------------------------------------------
link_schema = ArrayType(StructType([
    StructField("display_text",     StringType()),
    StructField("target_reference", StringType()),
    StructField("link_type",        StringType()),
]))

BRACKET_RE = re.compile(r"\[\[(.*?)\]\]")

def extract_links(content):
    if not content:
        return []
    out, seen = [], set()
    for match in BRACKET_RE.findall(content):
        if ">>" in match:
            display_text, target = match.split(">>", 1)
        else:
            display_text, target = match, match
        display_text = display_text.strip()
        parsed = parse_target(target)
        if parsed is None:
            continue
        link_type, target_reference = parsed
        if not target_reference:
            continue
        key = (target_reference, display_text)
        if key in seen:
            continue
        seen.add(key)
        out.append((display_text, target_reference, link_type))
    return out

extract_links_udf = F.udf(extract_links, link_schema)

# ----------------------------------------------------------
# Apply distributed, then explode to one row per link
# ----------------------------------------------------------
hyperlinks_df = (
    knowledge_nodes
    .select(
        F.col("node_id").alias("source_node"),
        F.col("node_name").alias("source_title"),
        F.col("node_content"),                 # kept for Cell 8 sentence matching
    )
    .withColumn("_links", extract_links_udf(F.col("node_content")))
    .withColumn("_link", F.explode_outer("_links"))
    .filter(F.col("_link").isNotNull())
    .select(
        "source_node",
        "source_title",
        "node_content",
        F.col("_link.display_text").alias("display_text"),
        F.col("_link.target_reference").alias("target_reference"),
        F.col("_link.link_type").alias("link_type"),
    )
    .cache()
)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------
total_links = hyperlinks_df.count()

print()
print("=" * 80)
print("HYPERLINK SUMMARY")
print("=" * 80)
print(f"Knowledge Nodes : {knowledge_nodes.count()}")
print(f"Hyperlinks      : {total_links}")
print("(Passed to cell 8 for sentence matching)")
print()
print("Link types found:")
display(hyperlinks_df.groupBy("link_type").count().orderBy(F.desc("count")))
print()
display(hyperlinks_df.select("source_title", "target_reference", "display_text", "link_type").limit(10))

In [ ]:
# ==========================================================
# Stage 6 : Sentence Matching
# Cell 8 : Locate Sentence Containing Each Hyperlink (20k-safe)
# ==========================================================
import re
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

print("=" * 80)
print("SENTENCE MATCHING (Cell 8)")
print("=" * 80)

# ----------------------------------------------------------
# Split content into sentences
# ----------------------------------------------------------
def split_into_sentences(content):
    if not content:
        return []
    content = re.sub(r'\{\{code\}\}.*?\{\{/code\}\}', ' ', content, flags=re.DOTALL)
    content = re.sub(r'\{\{[^}]*?\}\}', ' ', content)        # macros
    content = re.sub(r'\(\%.*?\%\)', ' ', content)           # style annotations
    content = re.sub(r'==+', '.', content)                   # headers -> boundary
    content = re.sub(r'\\\\', ' ', content)                  # wiki line breaks
    sentences = re.split(r'(?<=[.!?])\s+', content)
    return [s.strip() for s in sentences if s and len(s.strip()) > 10]

# ----------------------------------------------------------
# Clean wiki markup from a sentence
# ----------------------------------------------------------
def clean_sentence(sentence):
    s = re.sub(r'\[\[(.*?)\>\>.*?\]\]', r'\1', sentence)     # [[Display>>Target]] -> Display
    s = re.sub(r'\[\[(.*?)\]\]', r'\1', s)                   # [[X]] -> X
    s = re.sub(r'\|\|[^\s]*', '', s)                         # drop ||anchor
    s = s.replace('(((', '').replace(')))', '')
    s = s.replace('**', '').replace('*', '')
    s = re.sub(r'\(\s*\)', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# ----------------------------------------------------------
# parse_target: self-contained (executors can't see Cell 7)
# ----------------------------------------------------------
def _normalize_reference(ref):
    if not ref:
        return None
    ref = ref.strip()
    if "||" in ref:
        ref = ref.split("||")[0]
    if "#" in ref:
        ref = ref.split("#")[0]
    ref = ref.replace("&amp;", "&")
    ref = re.sub(r"\s+", "", ref)
    return ref.strip()

def _parse_target(target):
    if not target:
        return None
    target = target.strip()
    if target.startswith("doc:wiki:"):
        return ("doc:wiki", _normalize_reference(target[len("doc:wiki:"):]))
    if target.startswith("doc:Temp_Import"):
        return ("doc:Temp_Import", _normalize_reference(target[len("doc:"):]))
    if target.startswith("Temp_Import"):
        return ("bare:Temp_Import", _normalize_reference(target))
    return None

# ----------------------------------------------------------
# UDF: from (node_id, content) -> array of sentence-link records
# ----------------------------------------------------------
rec_schema = ArrayType(StructType([
    StructField("target_reference", StringType()),
    StructField("display_text",     StringType()),
    StructField("link_type",        StringType()),
    StructField("sentence",         StringType()),
]))

BRACKET_RE = re.compile(r"\[\[(.*?)\]\]")

def match_sentences(node_id, content):
    if not content:
        return []
    out = []
    for sentence in split_into_sentences(content):
        matches = BRACKET_RE.findall(sentence)
        if not matches:
            continue
        seen = set()
        cleaned = clean_sentence(sentence)
        for raw in matches:
            if ">>" in raw:
                display_text, target = raw.split(">>", 1)
            else:
                display_text, target = raw, raw
            display_text = display_text.strip()
            parsed = _parse_target(target)
            if parsed is None:
                continue
            link_type, target_reference = parsed
            if not target_reference:
                continue
            if target_reference == node_id:          # drop self-loops
                continue
            key = (target_reference, cleaned)
            if key in seen:
                continue
            seen.add(key)
            out.append((target_reference, display_text, link_type, cleaned))
    return out

match_sentences_udf = F.udf(match_sentences, rec_schema)

# ----------------------------------------------------------
# Apply distributed + explode
# ----------------------------------------------------------
sentences_df = (
    knowledge_nodes
    .select(
        F.col("node_id").alias("source_node"),
        F.col("node_name").alias("source_title"),
        F.col("node_content"),
    )
    # after the rename above, the column is "source_node", not "node_id"
    .withColumn("_recs", match_sentences_udf(F.col("source_node"), F.col("node_content")))
    .withColumn("_rec", F.explode_outer("_recs"))
    .filter(F.col("_rec").isNotNull())
    .select(
        "source_node",
        "source_title",
        F.col("_rec.target_reference").alias("target_reference"),
        F.col("_rec.display_text").alias("display_text"),
        F.col("_rec.link_type").alias("link_type"),
        F.col("_rec.sentence").alias("sentence"),
    )
    .cache()
)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------
total_recs = sentences_df.count()

print()
print("=" * 80)
print("SENTENCE MATCHING SUMMARY")
print("=" * 80)
print(f"Knowledge Nodes  : {knowledge_nodes.count()}")
print(f"Sentence Records : {total_recs}")
print("(Ready for Cell 9 LLM Semantic Extraction)")
print()
display(sentences_df.select("source_title", "target_reference", "display_text", "sentence").limit(10))

In [ ]:
LLM call

In [ ]:
# ==========================================================
# Stage 7 : Semantic Relationship Extraction
# Cell 9 : Extract Semantic Predicate (hardened)
# ==========================================================
import re
import time
import random
import concurrent.futures
from pyspark.sql import Row
from pyspark.sql import functions as F

print("=" * 80)
print("SEMANTIC RELATIONSHIP EXTRACTION")
print("=" * 80)

# ----------------------------------------------------------
# System prompt (your prompt, unchanged — it's good)
# ----------------------------------------------------------
SYSTEM_PROMPT = """You are an advanced NLP system specialized in Knowledge Graph construction for technical wiki documents.
Your task is to analyze a single sentence and extract the precise semantic relationship (Predicate) connecting the Source Node to the Target Node.

Rules for Extraction:
1. Return EXACTLY ONE predicate. Do not write full sentences or explanations.
2. Format as lowercase snake_case (e.g., depends_on, uses, part_of, defines, owned_by).
3. The Target is usually an XWiki reference starting with "doc:" or "doc:wiki:". The sentence will contain this exact reference.
4. You must infer the grammatical relationship between the Source and this Target reference.
5. Avoid generic predicates like 'mentions' or 'related_to'. Look for the specific action or relationship in the sentence (e.g. "detailed in", "owned by").
6. Do NOT include any punctuation, quotes, or markdown. Just the snake_case string.

Examples:

Source: Turbine Engineering
Target: HotEnd Design
Sentence: Turbine Engineering is a team within HotEnd Design.
Predicate: within

Source: MTCCM Help
Target: MTCCM Team
Sentence: This artefact is owned by the MTCCM team.
Predicate: owned_by

Source: OpenTCCM
Target: MATLAB
Sentence: OpenTCCM uses MATLAB to calculate thrust.
Predicate: uses

Source: Endurance Test Blocking
Target: Blocking Rules
Sentence: This rule is detailed in the Blocking Rules document.
Predicate: detailed_in
"""

# ----------------------------------------------------------
# Config
# ----------------------------------------------------------
MAX_THREADS   = 15
MAX_RETRIES   = 4        # retry on rate limit / transient errors
BACKOFF_BASE  = 1.5      # exponential backoff base (seconds)

# in-process cache: identical (source,target,sentence) -> predicate
_predicate_cache = {}

# ----------------------------------------------------------
# Worker with retry + backoff + distinct failure marker
# ----------------------------------------------------------
def call_llm_with_retry(user_prompt):
    for attempt in range(MAX_RETRIES):
        try:
            response = llm.invoke([
                ("system", SYSTEM_PROMPT),
                ("human", user_prompt),
            ])
            return response.content, None
        except Exception as e:
            msg = str(e).lower()
            # retry on throttling / transient; otherwise bail
            if ("429" in msg or "rate" in msg or "timeout" in msg
                    or "503" in msg or "overloaded" in msg):
                sleep = (BACKOFF_BASE ** attempt) + random.uniform(0, 0.5)
                time.sleep(sleep)
                continue
            return None, e          # non-retryable
    return None, Exception("max_retries_exhausted")

def process_record(record):
    source_title = record["source_title"]
    target_title = record["target_title"] or record["display_text"]  # prefer real title
    sentence     = record["sentence"]

    cache_key = (source_title, target_title, sentence)
    if cache_key in _predicate_cache:
        predicate = _predicate_cache[cache_key]
    else:
        user_prompt = (f"Source: {source_title}\n"
                       f"Target: {target_title}\n"
                       f"Sentence: {sentence}\n"
                       f"Predicate:")
        content, err = call_llm_with_retry(user_prompt)

        if err is not None:
            predicate = "extraction_failed"        # DISTINCT from related_to
        else:
            predicate = (content or "").strip().lower()
            predicate = re.sub(r'[^a-z_]', '', predicate.replace(' ', '_'))
            if not predicate or len(predicate) > 30:
                predicate = "related_to"           # genuine low-confidence

        _predicate_cache[cache_key] = predicate

    return Row(
        subject=source_title,
        predicate=predicate,
        object=target_title,
        subject_id=record["source_node"],
        object_id=record["target_reference"],
        sentence=sentence,
        link_type=record["link_type"],
    )

# ----------------------------------------------------------
# Join in the TARGET title (so LLM sees a real name, not a raw label)
# ----------------------------------------------------------
target_titles = knowledge_nodes.select(
    F.col("node_id").alias("target_reference"),
    F.col("node_name").alias("target_title"),
)

sentences_enriched = (
    sentences_df
    .join(target_titles, on="target_reference", how="left")
)

# ----------------------------------------------------------
# Concurrent execution
# NOTE (20k): collect()+threads works to a few thousand records.
# For full 20k, switch to Spark mapInPandas so calls run on
# executors with per-partition rate limiting. Ask when you get there.
# ----------------------------------------------------------
def extract_all_predicates(df, max_threads=MAX_THREADS):
    records = df.collect()
    total = len(records)
    print(f"Extracting predicates for {total} relationships "
          f"({max_threads} threads)...\n")
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as ex:
        futures = {ex.submit(process_record, r): r for r in records}
        count = 0
        for fut in concurrent.futures.as_completed(futures):
            count += 1
            results.append(fut.result())
            if count % 10 == 0 or count == total:
                print(f"Processed {count} / {total}")
    return results

semantic_records = extract_all_predicates(sentences_enriched)
rdf_df = spark.createDataFrame(semantic_records).cache()

# ----------------------------------------------------------
# Summary + quality signal
# ----------------------------------------------------------
print()
print("=" * 80)
print("PREDICATE DISTRIBUTION")
print("=" * 80)
display(rdf_df.groupBy("predicate").count().orderBy(F.desc("count")))

print()
print("=" * 80)
print("SAMPLE SEMANTIC TRIPLES")
print("=" * 80)
display(rdf_df.select("subject", "predicate", "object", "sentence").limit(15))

In [ ]:
# ==========================================================
# Stage 8 : RDF Triple Generation
# Cell 10 : Convert Semantic Records to Unique RDF Triples
# ==========================================================
from pyspark.sql import functions as F

print("=" * 80)
print("RDF TRIPLE GENERATION (Cell 10)")
print("=" * 80)

# predicates that are NOT real relationships -> exclude from graph
NON_SEMANTIC = ["extraction_failed"]     # keep 'related_to' (weak but real)

validated_rdf_df = (
    rdf_df
    # 1. Skip incomplete triples
    .filter(
        F.col("subject_id").isNotNull() & (F.col("subject_id") != "") &
        F.col("predicate").isNotNull()  & (F.col("predicate")  != "") &
        F.col("object_id").isNotNull()  & (F.col("object_id")  != "")
    )
    # 2. Drop failed extractions (LLM errors, not real edges)
    .filter(~F.col("predicate").isin(NON_SEMANTIC))
    # 3. Skip self-loops
    .filter(F.col("subject_id") != F.col("object_id"))
    # 4. Deduplicate on the RDF triple key
    .dropDuplicates(["subject_id", "predicate", "object_id"])
    # 5. Reference-based subject/object; titles for display
    .select(
        F.col("subject_id").alias("subject"),         # XML reference (node key)
        F.col("predicate"),
        F.col("object_id").alias("object"),           # XML reference (node key)
        F.col("subject").alias("subject_title"),      # display title
        F.col("object").alias("object_title"),        # display title
        F.col("sentence"),
        F.col("link_type"),
    )
    .cache()
)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------
output_count = validated_rdf_df.count()      # materializes cache
input_count  = rdf_df.count()                # cached from Cell 9

predicate_distribution = (
    validated_rdf_df.groupBy("predicate").count().orderBy(F.desc("count"))
)

print()
print("=" * 80)
print("RDF TRIPLE SUMMARY")
print("=" * 80)
print(f"Semantic Records (input) : {input_count}")
print(f"Unique RDF Triples       : {output_count}")
print(f"Dropped (failed/self/dup): {input_count - output_count}")
print()
print("-" * 80)
print("PREDICATE DISTRIBUTION")
print("-" * 80)
for row in predicate_distribution.collect():
    print(f"{row['predicate']:<25} : {row['count']}")

print()
print("-" * 80)
print("SAMPLE RDF TRIPLES (reference-based)")
print("-" * 80)
for row in validated_rdf_df.limit(10).collect():
    print(f" Subject   : {row['subject']}")
    print(f" Predicate : {row['predicate']}")
    print(f" Object    : {row['object']}")
    print(f" Display   : ({row['subject_title']}, {row['predicate']}, {row['object_title']})")
    print()

display(validated_rdf_df)

In [ ]:
# ==========================================================
# Stage 9 : Knowledge Graph Construction
# Cell 11 : Build Knowledge Graph from RDF Triples
# ==========================================================
import networkx as nx

print("=" * 80)
print("KNOWLEDGE GRAPH CONSTRUCTION (Cell 11)")
print("=" * 80)

# ----------------------------------------------------------
# 1. Directed graph.
#    MultiDiGraph keeps parallel edges (A uses B AND A references B).
#    Use nx.DiGraph() instead if you want at most one edge per pair.
# ----------------------------------------------------------
KG = nx.MultiDiGraph()

# ----------------------------------------------------------
# 2. Add ALL documents as nodes FIRST (so isolated docs survive,
#    and every node carries title + content + source for retrieval)
# ----------------------------------------------------------
print("Adding all knowledge nodes (with content)...")
all_nodes = knowledge_nodes.select(
    "node_id", "node_name", "node_content", "source"
).collect()

for r in all_nodes:
    KG.add_node(
        r["node_id"],
        title=r["node_name"],
        content=r["node_content"],   # needed by Context Builder (Cell 17)
        source=r["source"],
    )

print(f"Base nodes added: {KG.number_of_nodes()}")

# ----------------------------------------------------------
# 3. Add edges from validated triples
# ----------------------------------------------------------
print("Collecting validated RDF triples...")
validated_triples = validated_rdf_df.collect()
print("Adding semantic edges...\n")

for row in validated_triples:
    subj = row["subject"]
    obj  = row["object"]

    # target may be a doc we didn't load (name/GUID not in node set)
    # -> add a lightweight node so the edge is valid
    if subj not in KG:
        KG.add_node(subj, title=row["subject_title"], content=None, source=None)
    if obj not in KG:
        KG.add_node(obj, title=row["object_title"], content=None, source=None)

    KG.add_edge(
        subj,
        obj,
        predicate=row["predicate"],
        sentence=row["sentence"],
        link_type=row["link_type"],
    )

# ----------------------------------------------------------
# 4. Graph metrics (your retriever notes want degree; add it now)
# ----------------------------------------------------------
for n in KG.nodes():
    KG.nodes[n]["degree"] = KG.degree(n)

# ----------------------------------------------------------
# 5. Summary
# ----------------------------------------------------------
print("=" * 80)
print("KNOWLEDGE GRAPH SUMMARY")
print("=" * 80)
print(f"Total Nodes : {KG.number_of_nodes()}")
print(f"Total Edges : {KG.number_of_edges()}")
print(f"Triples in  : {len(validated_triples)}   (edges should match if MultiDiGraph)")

# isolated nodes = docs with no links
isolated = [n for n in KG.nodes() if KG.degree(n) == 0]
print(f"Isolated docs (no links): {len(isolated)}")

print()
print("-" * 80)
print("SAMPLE NODES (Reference -> Title)")
print("-" * 80)
for node, attr in list(KG.nodes(data=True))[:10]:
    print(f" Node  : {node}")
    print(f" Title : {attr.get('title', 'N/A')}  | degree={attr.get('degree',0)}\n")

print("-" * 80)
print("SAMPLE EDGES (Subject --[predicate]--> Object)")
print("-" * 80)
for s, t, attr in list(KG.edges(data=True))[:10]:
    st = KG.nodes[s].get("title", s)
    tt = KG.nodes[t].get("title", t)
    print(f" Display  : ({st}) --[{attr.get('predicate','?')}]--> ({tt})")
    print(f" Internal : ({s}) -> ({t})\n")

print("=" * 80)
print("Knowledge Graph Successfully Created!")
print("=" * 80)

In [ ]:
# ==========================================================
# Cell 12 : Visualize Knowledge Graph — ALL nodes + edges + predicates
# ==========================================================
import matplotlib.pyplot as plt
import networkx as nx

print("=" * 80)
print("KNOWLEDGE GRAPH VISUALIZATION")
print("=" * 80)
print(f"Total Nodes : {KG.number_of_nodes()}")
print(f"Total Edges : {KG.number_of_edges()}")

# ----------------------------------------------------------
# Labels: real docs -> title ; shell nodes -> short id (so GUIDs
# don't form text-walls). Predicates shown on ALL edges.
# ----------------------------------------------------------
node_labels = {}
node_colors = []
node_sizes  = []
degrees = dict(KG.degree())

for node, attr in KG.nodes(data=True):
    if attr.get("content") is not None:      # loaded doc
        node_labels[node] = (attr.get("title") or node)[:18]
        node_colors.append("#f4a261")
        node_sizes.append(1600 + degrees[node]*200)
    else:                                     # shell node
        node_labels[node] = (attr.get("title") or node)[:10]
        node_colors.append("#bcdff1")
        node_sizes.append(400 + degrees[node]*60)

edge_labels = {}
for u, v, attr in KG.edges(data=True):
    edge_labels[(u, v)] = attr.get("predicate", "")

# ----------------------------------------------------------
# Big canvas + strong spread so all 201 nodes separate
# ----------------------------------------------------------
plt.figure(figsize=(40, 30))           # very large so everything fits
pos = nx.spring_layout(KG, seed=42, k=3.5, iterations=250)

nx.draw_networkx_nodes(KG, pos, node_size=node_sizes,
                       node_color=node_colors, edgecolors="black", linewidths=0.5)
nx.draw_networkx_labels(KG, pos, labels=node_labels,
                        font_size=6, font_weight="bold")
nx.draw_networkx_edges(KG, pos, arrows=True, arrowsize=8,
                       width=0.6, edge_color="gray", alpha=0.4)

# ALL predicate labels (will be dense — open the saved PNG and zoom)
nx.draw_networkx_edge_labels(KG, pos, edge_labels=edge_labels,
                             font_size=5, rotate=False,
                             bbox=dict(boxstyle="round,pad=0.05",
                                       fc="white", ec="none", alpha=0.6))

plt.title("Semantic Knowledge Graph — all nodes, edges & predicates",
          fontsize=24, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig("kg_full_all.png", dpi=150, bbox_inches="tight")   # save hi-res
plt.show()

# %pip install pyvis
from pyvis.network import Network

net = Network(height="900px", width="100%", directed=True, bgcolor="#ffffff")
net.barnes_hut(gravity=-15000, spring_length=300)

degrees = dict(KG.degree())
for n, d in KG.nodes(data=True):
    loaded = d.get("content") is not None
    net.add_node(
        n,
        label=(d.get("title") or n)[:22],          # ALL nodes labelled
        title=f"{d.get('title')}\n{n}\ndegree={degrees[n]}",
        size=14+degrees[n]*3 if loaded else 6,
        color="#f4a261" if loaded else "#bcdff1",
    )

for u, v, d in KG.edges(data=True):
    net.add_edge(u, v,
                 label=d.get("predicate", ""),      # predicate on EVERY edge
                 title=d.get("sentence", ""))

net.write_html("kg_full.html")
with open("kg_full.html") as f:
    displayHTML(f.read())
print("Saved: kg_full_all.png  — open and ZOOM to read predicates")

In [ ]:
# ==========================================================
# Stage 12 : GraphRAG Online Phase (Retriever)
# Cell 16 : Query Understanding + Traversal + Top-K Ranking
# ==========================================================
import re

print("=" * 80)
print("GRAPHRAG RETRIEVER")
print("=" * 80)

USER_QUERY = "How can a new Test Environment be created in the application?"
TOP_K = 10
print(f"User Query: {USER_QUERY}\n")

STOPWORDS = {"how","can","a","an","new","be","created","create","in","the","what",
             "is","of","to","and","for","on","it","does","typically","why","are",
             "with","this","that","from","by","as","at","or","we","i","you"}

def get_keywords(text):
    return set(re.findall(r'[a-z0-9]+', str(text).lower())) - STOPWORDS

query_keywords = get_keywords(USER_QUERY)
print(f"Query keywords: {query_keywords}\n")

# ----------------------------------------------------------
# 1. Find start nodes — match against title AND content
# ----------------------------------------------------------
node_scores = []
for node, attr in KG.nodes(data=True):
    title   = attr.get("title", "") or ""
    content = attr.get("content", "") or ""
    # title matches weigh more than content matches
    t_overlap = len(query_keywords & get_keywords(title))
    c_overlap = len(query_keywords & get_keywords(content[:2000]))  # cap for speed
    score = t_overlap * 3 + c_overlap
    if score > 0:
        node_scores.append((node, title, score))

node_scores.sort(key=lambda x: x[2], reverse=True)
start_nodes = [n[0] for n in node_scores[:3]]

print(f"Start nodes: {len(start_nodes)}")
for n, t, s in node_scores[:3]:
    print(f"  - {t}  (score={s})")
print()

# ----------------------------------------------------------
# 2. Traversal — collect nodes with hop distance
#    follows BOTH directions (successors + predecessors)
# ----------------------------------------------------------
def neighbors_both(node):
    return set(KG.successors(node)) | set(KG.predecessors(node))

collected = {}     # node -> best (smallest) hop distance
retrieved_edges = []
visited_edges = set()

def explore(frontier, hop):
    nxt = set()
    for cur in frontier:
        for nb in neighbors_both(cur):
            # record edge (MultiDiGraph-safe: pull predicate from any parallel edge)
            for a, b in [(cur, nb), (nb, cur)]:
                if KG.has_edge(a, b):
                    ekey = (a, b)
                    if ekey not in visited_edges:
                        visited_edges.add(ekey)
                        ed = KG.get_edge_data(a, b)
                        # MultiDiGraph -> dict-of-dicts; take first parallel edge
                        first = ed[list(ed.keys())[0]] if ed else {}
                        retrieved_edges.append((a, b, first))
            if nb not in collected:
                collected[nb] = hop
                nxt.add(nb)
    return nxt

# seed start nodes at hop 0
for s in start_nodes:
    collected[s] = 0

frontier = set(start_nodes)
frontier = explore(frontier, 1)                 # hop 1
frontier = explore(frontier, 2)                 # hop 2
if len(retrieved_edges) < 5:                     # fallback hop 3
    print("[expanding to hop 3 — sparse context]\n")
    explore(frontier, 3)

print(f"Traversal: {len(collected)} nodes, {len(retrieved_edges)} edges\n")

# ----------------------------------------------------------
# 3. Rank collected nodes -> TOP-K documents
#    score = keyword relevance + closeness (fewer hops) + degree
# ----------------------------------------------------------
def node_relevance(node):
    a = KG.nodes[node]
    kw = len(query_keywords & get_keywords((a.get("title","") or "") + " " +
                                           (a.get("content","") or "")[:2000]))
    hop = collected.get(node, 99)
    deg = a.get("degree", KG.degree(node))
    # closer hops and more keyword hits score higher; degree is a small boost
    return kw * 5 + (3 - min(hop, 3)) * 2 + deg * 0.1

ranked = sorted(collected.keys(), key=node_relevance, reverse=True)

# only return docs that actually have content (skip shell nodes)
candidates = []
for node in ranked:
    a = KG.nodes[node]
    if a.get("content"):                         # has real content
        candidates.append({
            "doc_id": node,
            "title": a.get("title", node),
            "reference": node,
            "content": a.get("content", ""),
            "hop": collected.get(node, 99),
            "score": round(node_relevance(node), 2),
        })
    if len(candidates) >= TOP_K:
        break

# ----------------------------------------------------------
# 4. Display top-K
# ----------------------------------------------------------
print("=" * 80)
print(f"TOP {TOP_K} RETRIEVED DOCUMENTS")
print("=" * 80)
if not candidates:
    print("No content-bearing documents found for this query.")
for rank, c in enumerate(candidates, 1):
    print(f"\n{'-'*80}")
    print(f" Rank      : {rank}")
    print(f" Title     : {c['title']}")
    print(f" Reference : {c['reference']}")
    print(f" Hop       : {c['hop']}")
    print(f" Score     : {c['score']}")
    print(f" Content   : {c['content'][:300]}...")

print(f"\n{'='*80}")
print(f"Total retrieved: {len(candidates)} documents")

In [ ]:
# ==========================================================
# Stage 12 : GraphRAG Online Phase
# Cell 17 : Context Builder (top-K docs + graph relationships)
# ==========================================================
import re

print("=" * 80)
print("GRAPHRAG CONTEXT BUILDER")
print("=" * 80)

MAX_CHARS_PER_DOC = 1500      # cap so we don't blow the LLM context window

# ----------------------------------------------------------
# 1. Document text from the TOP-K retrieved candidates (Cell 16)
#    (not just start_nodes — this is the GraphRAG payoff)
# ----------------------------------------------------------
node_texts = []
for c in candidates:                      # <-- uses Cell 16's ranked top-10
    title   = c["title"]
    content = (c["content"] or "")[:MAX_CHARS_PER_DOC]
    node_texts.append(
        f"DOCUMENT TITLE: {title}\n"
        f"REFERENCE: {c['reference']}\n"
        f"CONTENT:\n{content}"
    )
FULL_NODE_CONTEXT = "\n\n".join(node_texts) if node_texts else ""

# ----------------------------------------------------------
# 2. Readable graph relationships from traversed edges
# ----------------------------------------------------------
def clean_title(node, raw_title):
    t = str(raw_title)
    if "doc:" in t or "WebHome" in t or "Temp_Import" in t:
        t = t.split(".")[-1].replace("WebHome", "").strip()
        t = re.sub(r'[^a-zA-Z0-9\s]', ' ', t).strip()
        if not t:
            t = "an external document"
    return t

edge_lines = []
for idx, (u, v, attr) in enumerate(retrieved_edges, 1):
    src_title = clean_title(u, KG.nodes[u].get("title", u))
    tgt_title = clean_title(v, KG.nodes[v].get("title", v))
    predicate = str(attr.get("predicate", "related to")).replace("_", " ")
    edge_lines.append(f"{idx}. {src_title} {predicate} {tgt_title}.")

GRAPH_EDGE_CONTEXT = "\n".join(edge_lines) if edge_lines else ""

print(f"Context built: {len(candidates)} docs, {len(edge_lines)} relationships")

# ----------------------------------------------------------
# 3. Prompt construction
# ----------------------------------------------------------
SYSTEM_PROMPT = """You are an expert technical assistant.
CRITICAL RULES:
1. Answer the user's question ONLY using the retrieved Knowledge Graph context below. Do NOT use outside training knowledge.
2. If both 'GRAPH RELATIONSHIPS' and 'RAW DOCUMENT TEXT' are empty, reply exactly: "The information is unavailable in the Knowledge Graph."
3. If context IS present, be analytical. Do not just say the exact answer is missing. Synthesize a detailed answer from whatever relationships and text are available.
4. Make logical inferences from the edges (e.g. if A 'opens' B, or A 'depends on' B, explain how they relate). Explain what the graph shows about the topic."""

FINAL_PROMPT = f"""
Question:
{USER_QUERY}
---------------------------------------
RAW DOCUMENT TEXT:
{FULL_NODE_CONTEXT}
---------------------------------------
GRAPH RELATIONSHIPS:
{GRAPH_EDGE_CONTEXT}
"""

print("Prompt constructed. Ready for Cell 18 (LLM).")
print(f"\nApprox prompt length: {len(FINAL_PROMPT)} chars")

In [ ]:
# ==========================================================
# Cell 18 : Top-K CHUNK Retrieval from Knowledge Graph (No LLM)
# BM25 scoring + graph traversal + layered dedup
# Output: {"predictions": [{"data": [{content, title, reference}]}]}
# ==========================================================
import re
import math
import json
import hashlib
from collections import Counter

query = """
Why is being a SQEP important in the context described in the document?
"""

TOP_K              = 10
CANDIDATE_NODES    = 80
CHUNK_SIZE         = 900      # smaller = more precise passages
CHUNK_STRIDE       = 700      # sliding window step (overlap = 200)
MIN_CHUNK_CHARS    = 120
MAX_CHUNKS_PER_DOC = 2
JACCARD_THRESHOLD  = 0.25
CONTAIN_THRESHOLD  = 0.50
SHINGLE_N          = 6

# BM25 parameters (standard values)
BM25_K1 = 1.5     # term-frequency saturation
BM25_B  = 0.75    # length normalization strength

GRAPH_BOOST   = 2.0   # weight of graph proximity
TITLE_BOOST   = 1.5   # weight of title match
PHRASE_BOOST  = 3.0   # weight of exact phrase hits

# ----------------------------------------------------------
# Tokenizing
# ----------------------------------------------------------
def tokens(text):
    return re.findall(r'[a-z0-9]+', str(text).lower())

def token_set(text):
    return set(tokens(text))

def shingles(text, n=SHINGLE_N):
    w = tokens(text)
    return {" ".join(w[i:i+n]) for i in range(max(0, len(w) - n + 1))}

def norm_hash(text):
    return hashlib.md5(" ".join(tokens(text)).encode()).hexdigest()

# ----------------------------------------------------------
# Cleaning
# ----------------------------------------------------------
def clean_text(t):
    t = t or ''
    t = re.sub(r'\(%[^%]*%\)', ' ', t)
    t = re.sub(r'\{\{[^}]*\}\}', ' ', t)
    t = re.sub(r'<[^>]{1,200}>', ' ', t)
    t = re.sub(r'\[\[([^\]>]+)>>[^\]]+\]\]', r'\1', t)
    t = re.sub(r'\(doc:[^)]+\)', ' ', t)
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'[ \t]{2,}', ' ', t)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

# ----------------------------------------------------------
# Chunking: sliding window over sentences (no mid-sentence cuts)
# ----------------------------------------------------------
def split_into_chunks(text, size=CHUNK_SIZE, stride=CHUNK_STRIDE):
    text = clean_text(text)
    if not text:
        return []
    # split to sentences/lines, then pack into windows
    units = [u.strip() for u in re.split(r'(?<=[.!?])\s+|\n+', text) if u.strip()]
    if not units:
        return []
    chunks, buf, start_idx = [], "", 0
    i = 0
    while i < len(units):
        buf, j = "", i
        while j < len(units) and len(buf) + len(units[j]) + 1 <= size:
            buf = (buf + " " + units[j]).strip()
            j += 1
        if j == i:                      # single oversized unit
            chunks.append(units[i][:size])
            i += 1
            continue
        if len(buf) >= MIN_CHUNK_CHARS:
            chunks.append(buf)
        # advance by stride worth of characters
        adv, k = 0, i
        while k < j and adv < stride:
            adv += len(units[k]) + 1
            k += 1
        i = max(k, i + 1)
    return chunks

# ----------------------------------------------------------
# STEP 1: build the chunk corpus ONCE (BM25 needs corpus stats)
# ----------------------------------------------------------
print("Building chunk corpus...")
CORPUS = []          # [{content, title, reference, tf, len}]
DF     = Counter()

for node, attr in KG.nodes(data=True):
    content = attr.get("content")
    if not content:
        continue
    title = attr.get("title", node)
    for ch in split_into_chunks(content):
        tf = Counter(tokens(ch))
        if not tf:
            continue
        CORPUS.append({
            "content":   ch,
            "title":     title,
            "reference": node,
            "tf":        tf,
            "len":       sum(tf.values()),
        })
        for w in tf:
            DF[w] += 1

N_CHUNKS = len(CORPUS)
AVG_LEN  = sum(c["len"] for c in CORPUS) / max(1, N_CHUNKS)

# BM25 IDF (with the standard +0.5 smoothing)
BM25_IDF = {
    w: math.log(1 + (N_CHUNKS - df + 0.5) / (df + 0.5))
    for w, df in DF.items()
}

print(f"Corpus: {N_CHUNKS} chunks | avg length {AVG_LEN:.0f} tokens "
      f"| vocab {len(DF)}\n")

# ----------------------------------------------------------
# STEP 2: graph traversal -> proximity score per document
# ----------------------------------------------------------
def graph_proximity(graph, q_terms):
    """Returns {node: hop_distance} from the best-matching seed nodes."""
    seeds = []
    for n, a in graph.nodes(data=True):
        if not a.get("content"):
            continue
        t_tok = token_set(a.get("title","") or "")
        c_tok = token_set((a.get("content","") or "")[:8000])
        s = (sum(BM25_IDF.get(w, 0) for w in q_terms if w in t_tok) * 3
             + sum(BM25_IDF.get(w, 0) for w in q_terms if w in c_tok))
        if s > 0:
            seeds.append((n, s))
    seeds.sort(key=lambda x: x[1], reverse=True)
    start = [n for n, _ in seeds[:5]]

    hops = {n: 0 for n in start}
    frontier = set(start)
    for hop in (1, 2):
        nxt = set()
        for cur in frontier:
            for nb in set(graph.successors(cur)) | set(graph.predecessors(cur)):
                if nb not in hops:
                    hops[nb] = hop
                    nxt.add(nb)
        frontier = nxt

    print(f"Graph: {len(start)} seed nodes -> {len(hops)} reachable nodes")
    return hops

# ----------------------------------------------------------
# STEP 3: BM25 score + graph boost + dedup
# ----------------------------------------------------------
def retrieve_chunks(graph, query_text, top_k=TOP_K):
    q_tok   = tokens(query_text)
    q_terms = set(q_tok)
    q_phr   = [" ".join(q_tok[i:i+2]) for i in range(len(q_tok)-1)]

    print("Top query terms by IDF:")
    for w in sorted(q_terms, key=lambda x: BM25_IDF.get(x, 0), reverse=True)[:6]:
        print(f"   {w:<20} {BM25_IDF.get(w, 0):.2f}"
              f"{'   <-- NOT IN CORPUS' if w not in BM25_IDF else ''}")
    print()

    hops = graph_proximity(graph, q_terms)

    scored = []
    for c in CORPUS:
        # --- BM25 core ---
        bm25 = 0.0
        for w in q_terms:
            f = c["tf"].get(w, 0)
            if f == 0:
                continue
            idf   = BM25_IDF.get(w, 0)
            denom = f + BM25_K1 * (1 - BM25_B + BM25_B * c["len"] / AVG_LEN)
            bm25 += idf * (f * (BM25_K1 + 1)) / denom
        if bm25 == 0:
            continue

        # --- graph proximity: closer to a seed = higher ---
        hop   = hops.get(c["reference"], 99)
        gprox = (3 - min(hop, 3)) / 3.0 * GRAPH_BOOST

        # --- title match ---
        t_tok = token_set(c["title"])
        tscore = sum(BM25_IDF.get(w, 0) for w in q_terms if w in t_tok) * TITLE_BOOST

        # --- exact phrase hits ---
        low = c["content"].lower()
        pscore = sum(1 for p in q_phr if p in low) * PHRASE_BOOST

        scored.append({
            "content":   c["content"],
            "title":     c["title"],
            "reference": c["reference"],
            "_score":    bm25 + gprox + tscore + pscore,
            "_bm25":     bm25,
            "_hop":      hop,
        })

    scored.sort(key=lambda x: x["_score"], reverse=True)

    # --- layered dedup during selection ---
    chosen, hashes, shing, toks, per_doc, dropped = [], set(), [], [], {}, 0
    for c in scored:
        if len(chosen) >= top_k:
            break
        if per_doc.get(c["reference"], 0) >= MAX_CHUNKS_PER_DOC:
            continue
        txt = c["content"]

        h = norm_hash(txt)
        if h in hashes:
            dropped += 1; continue

        sh = shingles(txt)
        if sh and any(len(sh & p) / len(sh | p) > JACCARD_THRESHOLD for p in shing):
            dropped += 1; continue

        ts = token_set(txt)
        if ts and any(len(ts & p) / min(len(ts), len(p)) > CONTAIN_THRESHOLD for p in toks):
            dropped += 1; continue

        chosen.append(c)
        hashes.add(h); shing.append(sh); toks.append(ts)
        per_doc[c["reference"]] = per_doc.get(c["reference"], 0) + 1

    print(f"Matched: {len(scored)} chunks | duplicates dropped: {dropped} "
          f"| selected: {len(chosen)}\n")
    return chosen

chunks = retrieve_chunks(KG, query, top_k=TOP_K)

# ----------------------------------------------------------
# Diagnostics — read this before trusting the scores
# ----------------------------------------------------------
print("-" * 70)
for i, c in enumerate(chunks, 1):
    print(f"{i:>2}. score={c['_score']:6.2f}  bm25={c['_bm25']:5.2f}  "
          f"hop={c['_hop']}  {c['title'][:45]}")
print("-" * 70, "\n")

# ----------------------------------------------------------
# Response payload
# ----------------------------------------------------------
response = {
    "predictions": [{
        "data": [
            {"content": c["content"], "title": c["title"], "reference": c["reference"]}
            for c in chunks
        ]
    }]
}

print("Status Code: 200\n")
print("Response:")
print(json.dumps(response, indent=2, ensure_ascii=False))

In [ ]:
LLM

In [ ]:
stage 12 gRAPhrag online phase reterive
cell 18 

print("=" * 80)
print("Graph Online: Final LLM Answer")
print("=" * 80)

print ("sending")

try: 
  response = llm.invoke([
    ("system", SYSTEM_PROMT),
    ("human",FINAL_PROMT)

    final_answer = response.content.strip()

    ---

    except Except as e:
    print(f"Error callin LLM:{e}")
  ])

In [ ]:
# ==========================================================
# Cell 28 : LLM-judged via MLflow  (Vector Search vs Knowledge Graph)
#
# BEFORE RUNNING, make these two one-line renames:
#   Cell 18 (KG)   : response = {"predictions": ...}   ->  response_kg = {...}
#   Cell 27 (VS)   : response = requests.post(...)     ->  http_vs   = requests.post(...)
# Nothing is pasted here - both payloads are fetched live.
# ==========================================================
import mlflow
import json
import requests
import pandas as pd
from mlflow.entities import Feedback, AssessmentSource
from databricks.agents.evals import metric, judges

K        = 10
QUESTION = "How do I open the OpenTCCM model using MTCCM?"

# ----------------------------------------------------------
# 1. VECTOR SEARCH  -> call the serving endpoint
# ----------------------------------------------------------
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
url   = "https://adb-3955688708870984.4.azuredatabricks.net/serving-endpoints/knowbot-retriever/invocations"

http_vs = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"dataframe_records": [{"query": QUESTION, "top_k": K}]},
    timeout=120,
)
http_vs.raise_for_status()
response_vs = http_vs.json()          # <- dict, straight from the endpoint

# ----------------------------------------------------------
# 2. KNOWLEDGE GRAPH -> already a dict from Cell 18
#    Cell 18 must have been run with the SAME question.
# ----------------------------------------------------------
# response_kg  comes from Cell 18

# ----------------------------------------------------------
# 3. Flatten to mlflow's schema: only 'doc_uri' and 'content'
# ----------------------------------------------------------
def to_chunks(payload, top_k=K):
    if isinstance(payload, dict) and "predictions" in payload:
        data = payload["predictions"][0].get("data", []) or []
    elif isinstance(payload, dict) and "data" in payload:
        data = payload["data"] or []
    elif isinstance(payload, list):
        data = payload
    else:
        data = []
    return [
        {"doc_uri": c.get("doc_uri") or c.get("reference", ""),
         "content": c.get("content", "") or ""}
        for c in data[:top_k]
    ]

print("vector search chunks  :", len(to_chunks(response_vs)))
print("knowledge graph chunks:", len(to_chunks(response_kg)))

# ----------------------------------------------------------
# 4. Custom LLM-judged Precision@K   (unchanged)
# ----------------------------------------------------------
def judged_precision_at_k(request, retrieved_context, k):
    """
    LLM-judged Precision@K: Uses MLflow's chunk_relevance judge to determine
    whether each retrieved chunk is semantically relevant to the query.
    """
    judged_precisions = [judges.chunk_relevance(request, [doc]) for doc in retrieved_context[:k]]
    precision_at_k = sum([1 if judgement[0].value == 'yes' else 0 for judgement in judged_precisions]) / k

    rationales = [
        f"""## Chunk {i+1}: `{retrieved_context[i].get('doc_uri', 'unknown')}`\n- **{judged_precisions[i][0].value}**: {judged_precisions[i][0].rationale}"""
        for i in range(len(judged_precisions))
    ]

    return Feedback(
        name=f'judged_precision_at_{k}',
        value=precision_at_k,
        rationale='\n'.join(rationales),
        source=AssessmentSource(source_type="LLM_JUDGE", source_id="chunk_relevance_judge")
    )

@metric
def judged_precision_at_10(request, retrieved_context):
    """Judged Precision@10 - fraction of top-10 chunks judged relevant by LLM."""
    return judged_precision_at_k(request=request, retrieved_context=retrieved_context, k=K)

# ----------------------------------------------------------
# 5. Build eval_data   (row 0 = vector search, row 1 = knowledge graph)
# ----------------------------------------------------------
eval_data_1 = [
    {"request": QUESTION, "retrieved_context": response_vs},
    {"request": QUESTION, "retrieved_context": response_kg},
]

eval_df = pd.DataFrame(eval_data_1)
eval_df["retrieved_context"] = eval_df["retrieved_context"].apply(to_chunks)

eval_data = (
    eval_df[["request", "retrieved_context"]]
    .to_dict(orient="records")
)

# ----------------------------------------------------------
# 6. Run MLflow evaluate
# ----------------------------------------------------------
with mlflow.start_run(run_name="graphrag_retrieval_eval_judged"):
    results = mlflow.evaluate(
        data=eval_data,
        model_type="databricks-agent",
        evaluator_config={
            "databricks-agent": {
                "metrics": ["chunk_relevance"]
            }
        },
        extra_metrics=[judged_precision_at_10]
    )

    print("=" * 60)
    print("    JUDGED PRECISION@K RESULTS (LLM-judged)")
    print("=" * 60)
    print(f"\nAll metrics: {results.metrics}")
    print("=" * 60)

# ----------------------------------------------------------
# 7. Per-system scores (results.metrics averages BOTH rows)
# ----------------------------------------------------------
eval_tbl = results.tables["eval_results"].copy()
eval_tbl.insert(0, "system", ["vector_search", "knowledge_graph"])
display(eval_tbl)

print("vector search chunks  :", len(response_vs["predictions"][0]["data"]))
print("knowledge graph chunks:", len(response_kg["predictions"][0]["data"]))

print("judging against:", QUESTION)
print("KG chunks:", len(to_chunks(response_kg)),
      " VS chunks:", len(to_chunks(response_vs)))